<a href="https://colab.research.google.com/github/gauwuhar/ieee-sist-2026/blob/main/Transcriptomic_hybrid.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Setup and Initial Imports

In [ ]:
import kagglehub
import numpy as np
import pandas as pd
import scipy.io
import scipy.sparse as sp
import torch
import torch.utils.data as data
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
import gc
import os
import shutil
from google.colab import files
from IPython.display import display, Image
import umap
import torch.optim as optim

#Kaggle Setup and Dataset Loading

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("amaishi/gse299623")

print("Path to dataset files:", path)

100%|██████████| 820M/820M [00:38<00:00, 22.4MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/amaishi/gse299623/versions/1


#Load and Filter Data

In [ ]:
# Load the sparse matrix and associated data
matrix_file = path + '/GSM9042132_A2_1_matrix.mtx/matrix.mtx'
features_file = path + '/GSM9042132_A2_1_features.tsv/features.tsv'
barcodes_file = path +'/GSM9042132_A2_1_barcodes.tsv/barcodes.tsv'


In [ ]:
# =========================
# Load sparse matrix + metadata
# =========================
counts = scipy.io.mmread(matrix_file).tocsr()
counts.data = counts.data.astype(np.float32, copy=False)

features = pd.read_csv(features_file, header=None, sep="\t")
features.columns = ["gene_id", "gene_name", "description"]

barcodes = pd.read_csv(barcodes_file, header=None, sep="\t")
barcodes.columns = ["barcode"]

print("Raw counts shape (genes x cells):", counts.shape)



Raw counts shape (genes x cells): (18082, 21070)


#Data Normalization and Feature Selection

In [ ]:
# =========================
# Basic filtering (memory-friendly: getnnz instead of (counts>0))
# =========================
min_cells = 10
min_genes = 200

gene_nnz = counts.getnnz(axis=1)        # #cells per gene
keep_genes = gene_nnz >= min_cells

cell_nnz = counts.getnnz(axis=0)        # #genes per cell
keep_cells = cell_nnz >= min_genes

counts = counts[keep_genes, :][:, keep_cells]

genes_kept = features.loc[keep_genes, "gene_name"].reset_index(drop=True)
barcodes_kept = barcodes.loc[keep_cells, "barcode"].reset_index(drop=True)

print("After filtering:", counts.shape)

After filtering: (16302, 21067)


In [ ]:
# =========================
# Normalize + log1p (NO diagonal matrix -> less RAM)
# =========================
library = np.asarray(counts.sum(axis=0)).ravel().astype(np.float32)
library[library == 0] = 1.0
scale = (1e4 / library).astype(np.float32)

# Multiply columns by scale (broadcast), stays sparse
X = counts.multiply(scale)
X.data = np.log1p(X.data).astype(np.float32, copy=False)

# You can delete counts now if you want to free more RAM
del counts
gc.collect()



3130

In [ ]:
# =========================
# HVG selection on sparse
# =========================
# Convert X to CSR format for efficient row slicing before calculating mean and var
X_csr = X.tocsr()

mean = np.asarray(X_csr.mean(axis=1)).ravel().astype(np.float32)
mean_sq = np.asarray(X_csr.multiply(X_csr).mean(axis=1)).ravel().astype(np.float32)
var = mean_sq - mean * mean

hvg = min(2000, X_csr.shape[0])

# Faster than full sort: pick top-k via argpartition
hvg_idx = np.argpartition(var, -hvg)[-hvg:]
# optional: sort selected genes by variance desc (just for nice ordering)
hvg_idx = hvg_idx[np.argsort(var[hvg_idx])[::-1]]

# cells x hvg, make CSR for fast row slicing
X_hvg = X_csr[hvg_idx, :].T.tocsr()
print("X_hvg shape (cells x hvg):", X_hvg.shape)

# Free big X if you only need X_hvg
del X, X_csr, mean, mean_sq, var
gc.collect()

X_hvg shape (cells x hvg): (21067, 2000)


0

In [ ]:
# =========================
# DataLoader that DOES NOT densify whole dataset
# - dataset returns indices
# - collate_fn converts only this batch (X_hvg[idxs]) to dense
# =========================
class IndexDataset(data.Dataset):
    def __init__(self, n):
        self.n = int(n)
    def __len__(self):
        return self.n
    def __getitem__(self, idx):
        return int(idx)

def make_sparse_batch_collate(X_csr):
    def collate(idxs):
        idxs = np.fromiter(idxs, dtype=np.int64)
        batch = X_csr[idxs].toarray()
        if batch.dtype != np.float32:
            batch = batch.astype(np.float32, copy=False)
        return torch.from_numpy(batch)
    return collate

batch_size = 32
dataset = IndexDataset(X_hvg.shape[0])
dl = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=True,
    drop_last=True,
    num_workers=0,                       # RAM-safe in Colab
    pin_memory=torch.cuda.is_available(),
    collate_fn=make_sparse_batch_collate(X_hvg),
)

In [ ]:
# =========================
# Loss (avoid pred[mask] indexing copies)
# =========================
def masked_mse(pred, target, mask):
    # mask: bool tensor, same shape
    diff2 = (pred - target).pow(2)
    mask_f = mask.float()
    denom = mask_f.sum().clamp(min=1.0)
    return (diff2 * mask_f).sum() / denom

In [ ]:





# =========================
# Model
# =========================
class GeneTransformer(nn.Module):
    def __init__(self, num_genes, d_model=64, nhead=4, num_layers=2, dropout=0.1):
        super().__init__()
        self.num_genes = num_genes
        self.d_model = d_model

        self.gene_emb = nn.Embedding(num_genes, d_model)
        self.val_proj = nn.Linear(1, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=4 * d_model,
            dropout=dropout,
            batch_first=True,
            activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.out = nn.Linear(d_model, 1)
        self.cls = nn.Parameter(torch.zeros(1, 1, d_model))

        # Cache gene ids once (saves per-forward allocations)
        self.register_buffer("gene_ids", torch.arange(num_genes), persistent=False)

    def forward(self, x):
        B, G = x.shape
        gene_ids = self.gene_ids.unsqueeze(0).expand(B, -1)  # (B, G)

        val = self.val_proj(x.unsqueeze(-1))                 # (B, G, d_model)
        pos = self.gene_emb(gene_ids)                        # (B, G, d_model)
        h = val + pos

        cls = self.cls.expand(B, 1, self.d_model)
        h = torch.cat([cls, h], dim=1)                       # (B, G+1, d_model)

        h = self.encoder(h)
        pred = self.out(h[:, 1:, :]).squeeze(-1)             # (B, G)
        cell_emb = h[:, 0, :]                                # (B, d_model)
        return pred, cell_emb



In [ ]:
# =========================
# Train
# =========================
device = "cuda" if torch.cuda.is_available() else "cpu"
num_genes = X_hvg.shape[1]   # IMPORTANT: match HVG count
model = GeneTransformer(num_genes=num_genes, d_model=64, nhead=4, num_layers=2).to(device)
opt = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)

accumulation_steps = 4
mask_ratio = 0.15
epochs = 10

use_amp = torch.cuda.is_available()
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

for epoch in range(epochs):
    model.train()
    opt.zero_grad(set_to_none=True)

    running_loss = 0.0
    n_updates = 0

    for step, x_true_batch in enumerate(dl):
        x_true_batch = x_true_batch.to(device, non_blocking=True)

        mask = (torch.rand_like(x_true_batch) < mask_ratio)
        x_masked = x_true_batch.masked_fill(mask, 0.0)

        with torch.cuda.amp.autocast(enabled=use_amp):
            pred, _ = model(x_masked)
            loss = masked_mse(pred, x_true_batch, mask) / accumulation_steps

        scaler.scale(loss).backward()

        if (step + 1) % accumulation_steps == 0:
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)

            running_loss += loss.item() * accumulation_steps
            n_updates += 1

    avg_loss = running_loss / max(1, n_updates)
    print(f"Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.4f}")

    gc.collect()

torch.save(model.state_dict(), "gene_transformer.pth")
print("Saved: gene_transformer.pth")

/tmp/ipython-input-2893321402.py:14: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
/tmp/ipython-input-2893321402.py:29: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


Epoch 1/10 | Loss: 0.9017
Epoch 2/10 | Loss: 0.7935
Epoch 3/10 | Loss: 0.7307
Epoch 4/10 | Loss: 0.6904
Epoch 5/10 | Loss: 0.6791
Epoch 6/10 | Loss: 0.6790
Epoch 7/10 | Loss: 0.6730
Epoch 8/10 | Loss: 0.6797
Epoch 9/10 | Loss: 0.6708
Epoch 10/10 | Loss: 0.6661
Saved: gene_transformer.pth


In [ ]:
model.eval()

# Create an evaluation DataLoader (no shuffling, no dropping last batch)
eval_dataset = IndexDataset(X_hvg.shape[0])
eval_dl = DataLoader(
    eval_dataset,
    batch_size=batch_size,
    shuffle=False,
    drop_last=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    collate_fn=make_sparse_batch_collate(X_hvg),
)

true_masked_values = []
predicted_masked_values = []

with torch.no_grad():
    for x_true_batch in eval_dl:
        x_true_batch = x_true_batch.to(device)

        # Generate a mask for evaluation (similar to training)
        mask = (torch.rand_like(x_true_batch) < mask_ratio)
        x_masked = x_true_batch.masked_fill(mask, 0.0)

        pred, _ = model(x_masked)

        # Collect true and predicted values only at masked positions
        true_masked_values.append(x_true_batch[mask].cpu().numpy())
        predicted_masked_values.append(pred[mask].cpu().numpy())

# Concatenate all collected values
all_true_masked = np.concatenate(true_masked_values)
all_predicted_masked = np.concatenate(predicted_masked_values)

# Calculate metrics
mse = mean_squared_error(all_true_masked, all_predicted_masked)
rmse = np.sqrt(mse)
mae = mean_absolute_error(all_true_masked, all_predicted_masked)
r2 = r2_score(all_true_masked, all_predicted_masked)

# Pearson correlation
pearson_corr, _ = scipy.stats.pearsonr(all_true_masked, all_predicted_masked)

print(f"Evaluation Metrics (on masked positions):")
print(f"  MSE: {mse:.4f}")
print(f"  RMSE: {rmse:.4f}")
print(f"  MAE: {mae:.4f}")
print(f"  R^2: {r2:.4f}")
print(f"  Pearson Correlation: {pearson_corr:.4f}")


Evaluation Metrics (on masked positions):
  MSE: 0.6640
  RMSE: 0.8149
  MAE: 0.6571
  R^2: 0.2254
  Pearson Correlation: 0.4752


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import math
import gc

# ---------- Better GeneTransformer ----------
class GeneTransformer(nn.Module):
    def __init__(self, num_genes, d_model=128, nhead=8, num_layers=4, dropout=0.1):
        super().__init__()
        self.num_genes = num_genes
        self.d_model = d_model

        self.gene_emb = nn.Embedding(num_genes, d_model)
        self.val_proj = nn.Linear(1, d_model)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=4*d_model,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
            norm_first=True,   # often helps stability
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)

        self.out = nn.Linear(d_model, 1)

        self.cls = nn.Parameter(torch.zeros(1, 1, d_model))
        self.mask_token = nn.Parameter(torch.zeros(1, 1, d_model))  # IMPORTANT: learnable [MASK]

        # init
        nn.init.normal_(self.cls, std=0.02)
        nn.init.normal_(self.mask_token, std=0.02)

    def forward(self, x):  # x: (B, G)
        B, G = x.shape
        gene_ids = torch.arange(G, device=x.device).unsqueeze(0).expand(B, G)

        val = self.val_proj(x.unsqueeze(-1))     # (B,G,d)
        pos = self.gene_emb(gene_ids)            # (B,G,d)
        h = val + pos

        # CLS
        cls = self.cls.expand(B, 1, self.d_model)
        h = torch.cat([cls, h], dim=1)           # (B,G+1,d)

        h = self.encoder(h)
        pred = self.out(h[:, 1:, :]).squeeze(-1) # (B,G)
        cell_emb = h[:, 0, :]
        return pred, cell_emb


# ---------- Loss on masked positions ----------
huber = nn.SmoothL1Loss(reduction="mean")  # Huber

def masked_huber(pred, target, mask):
    # pred/target: (B,G), mask: (B,G) bool
    if mask.sum() == 0:
        return huber(pred, target)
    return huber(pred[mask], target[mask])


# =========================
# Train (Improved)
# =========================
device = "cuda" if torch.cuda.is_available() else "cpu"

num_genes = X_hvg.shape[1]  # match HVG count
model = GeneTransformer(num_genes=num_genes, d_model=128, nhead=8, num_layers=4, dropout=0.1).to(device)

opt = optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-2, betas=(0.9, 0.95))

# scheduler: warmup + cosine
epochs = 30
mask_ratio = 0.20
accumulation_steps = 4
grad_clip = 1.0

steps_per_epoch = len(dl)
total_updates = math.ceil(steps_per_epoch / accumulation_steps) * epochs
warmup_updates = int(0.1 * total_updates)

def lr_schedule(update_idx):
    # linear warmup, then cosine decay
    if update_idx < warmup_updates:
        return (update_idx + 1) / max(1, warmup_updates)
    t = (update_idx - warmup_updates) / max(1, (total_updates - warmup_updates))
    return 0.5 * (1.0 + math.cos(math.pi * t))

use_amp = torch.cuda.is_available()
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

update_idx = 0

for epoch in range(epochs):
    model.train()
    opt.zero_grad(set_to_none=True)
    running_loss = 0.0
    n_updates = 0

    for step, x_true_batch in enumerate(dl):
        x_true_batch = x_true_batch.to(device, non_blocking=True)

        # OPTIONAL: do not mask zeros (zeros are common in scRNA and add noise)
        # mask only positions where target > 0
        base_mask = (x_true_batch > 0)
        rand_mask = (torch.rand_like(x_true_batch) < mask_ratio)
        mask = rand_mask & base_mask

        # make masked input
        x_masked = x_true_batch.clone()

        # replace masked positions not by 0, but by learnable token effect:
        # easiest: set value to 0, but ALSO inject mask_token later via adding to embeddings.
        # We can implement by setting to 0 here, and passing mask to forward if you want deeper integration.
        # Minimal improvement: set to 0 here (still ok) + we will add mask_token by "value shift"
        x_masked = x_masked.masked_fill(mask, 0.0)

        with torch.cuda.amp.autocast(enabled=use_amp):
            pred, _ = model(x_masked)

            # compute masked loss only
            loss = masked_huber(pred, x_true_batch, mask) / accumulation_steps

        scaler.scale(loss).backward()

        if (step + 1) % accumulation_steps == 0:
            # gradient clipping
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

            # update lr
            lr_mult = lr_schedule(update_idx)
            for pg in opt.param_groups:
                pg["lr"] = 5e-4 * lr_mult

            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)

            running_loss += loss.item() * accumulation_steps
            n_updates += 1
            update_idx += 1

    avg_loss = running_loss / max(1, n_updates)
    print(f"Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.4f} | LR: {opt.param_groups[0]['lr']:.2e}")
    gc.collect()

torch.save(model.state_dict(), "gene_transformer.pth")
print("Saved: gene_transformer.pth")


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
/tmp/ipython-input-60063179.py:93: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
/tmp/ipython-input-60063179.py:121: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


Epoch 1/30 | Loss: 0.4416 | LR: 1.66e-04
Epoch 2/30 | Loss: 0.1276 | LR: 3.31e-04
Epoch 3/30 | Loss: 0.1066 | LR: 4.97e-04
Epoch 4/30 | Loss: 0.1024 | LR: 4.98e-04
Epoch 5/30 | Loss: 0.1009 | LR: 4.94e-04
Epoch 6/30 | Loss: 0.1006 | LR: 4.85e-04
Epoch 7/30 | Loss: 0.1004 | LR: 4.74e-04
Epoch 8/30 | Loss: 0.0990 | LR: 4.60e-04
Epoch 9/30 | Loss: 0.0999 | LR: 4.43e-04
Epoch 10/30 | Loss: 0.0989 | LR: 4.23e-04
Epoch 11/30 | Loss: 0.0974 | LR: 4.01e-04
Epoch 12/30 | Loss: 0.0950 | LR: 3.77e-04
Epoch 13/30 | Loss: 0.0934 | LR: 3.51e-04
Epoch 14/30 | Loss: 0.0929 | LR: 3.24e-04
Epoch 15/30 | Loss: 0.0919 | LR: 2.96e-04
Epoch 16/30 | Loss: 0.0917 | LR: 2.68e-04
Epoch 17/30 | Loss: 0.0914 | LR: 2.39e-04
Epoch 18/30 | Loss: 0.0906 | LR: 2.10e-04
Epoch 19/30 | Loss: 0.0902 | LR: 1.82e-04
Epoch 20/30 | Loss: 0.0902 | LR: 1.54e-04
Epoch 21/30 | Loss: 0.0896 | LR: 1.28e-04
Epoch 22/30 | Loss: 0.0897 | LR: 1.04e-04
Epoch 23/30 | Loss: 0.0890 | LR: 8.15e-05
Epoch 24/30 | Loss: 0.0896 | LR: 6.14e-05
E

In [ ]:
import numpy as np
import torch
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import pearsonr

@torch.no_grad()
def eval_gene_transformer_masked(
    model,
    dl,
    device,
    mask_ratio=0.15,
    seed=42,
    max_batches=None,   # можно поставить 200 для быстрого теста
):
    model.eval()

    # чтобы маска была воспроизводимой
    g = torch.Generator(device=device)
    g.manual_seed(seed)

    y_true_all = []
    y_pred_all = []

    for b, batch in enumerate(dl):
        if max_batches is not None and b >= max_batches:
            break

        # batch может быть (x,) или просто x
        if isinstance(batch, (list, tuple)):
            x_true = batch[0]
        else:
            x_true = batch

        x_true = x_true.to(device, non_blocking=True)

        # mask (желательно не маскировать нули)
        base_mask = (x_true > 0)
        # FIX: torch.rand_like does not support generator, using torch.rand instead
        rand_vals = torch.rand(x_true.shape, device=x_true.device, generator=g)
        rand_mask = (rand_vals < mask_ratio)
        mask = rand_mask & base_mask

        x_masked = x_true.masked_fill(mask, 0.0)

        pred, _ = model(x_masked)

        # собираем только masked значения
        if mask.sum() > 0:
            y_true_all.append(x_true[mask].detach().cpu().numpy())
            y_pred_all.append(pred[mask].detach().cpu().numpy())

    if len(y_true_all) == 0:
        raise RuntimeError("Нет masked позиций. Увеличь mask_ratio или проверь данные (x_true > 0).")

    y_true = np.concatenate(y_true_all)
    y_pred = np.concatenate(y_pred_all)

    mse = mean_squared_error(y_true, y_pred)
    rmse = float(np.sqrt(mse))
    mae = mean_absolute_error(y_true, y_pred)

    # R^2 (если variance не нулевая)
    if np.std(y_true) > 1e-12:
        r2 = r2_score(y_true, y_pred)
    else:
        r2 = np.nan

    pear = pearsonr(y_true, y_pred)[0]

    print("Evaluation Metrics (on masked positions):")
    print(f"  MSE: {mse:.4f}")
    print(f"  RMSE: {rmse:.4f}")
    print(f"  MAE: {mae:.4f}")
    if np.isnan(r2):
        print("  R^2: Not applicable (true values are constant)")
    else:
        print(f"  R^2: {r2:.4f}")
    print(f"  Pearson Correlation: {pear:.4f}")

    return {
        "mse": mse,
        "rmse": rmse,
        "mae": mae,
        "r2": r2,
        "pearson": pear,
        "n_masked": int(y_true.shape[0]),
    }

# ====== пример вызова ======
device = "cuda" if torch.cuda.is_available() else "cpu"
_ = eval_gene_transformer_masked(
    model=model,
    dl=dl,
    device=device,
    mask_ratio=0.15,
    seed=42,
    max_batches=None
)

Evaluation Metrics (on masked positions):
  MSE: 0.1831
  RMSE: 0.4279
  MAE: 0.3297
  R^2: 0.6484
  Pearson Correlation: 0.8055


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import pearsonr

device = "cuda" if torch.cuda.is_available() else "cpu"

def make_mask(x_true, mask_ratio=0.15, seed=None):
    # x_true: torch.Tensor (B,G)
    if seed is not None:
        g = torch.Generator(device=x_true.device)
        g.manual_seed(seed)
        # FIX: torch.rand_like does not support generator
        rand = torch.rand(x_true.shape, device=x_true.device, generator=g)
    else:
        rand = torch.rand_like(x_true)
    base_mask = (x_true > 0)                 # не маскируем нули
    mask = (rand < mask_ratio) & base_mask
    x_masked = x_true.masked_fill(mask, 0.0)
    return x_masked, mask

def masked_mse(pred, target, mask):
    if mask.sum() == 0:
        return ((pred - target) ** 2).mean()
    return ((pred[mask] - target[mask]) ** 2).mean()

@torch.no_grad()
def eval_masked_metrics(model, dl, mask_ratio=0.15, seed=42, max_batches=None):
    model.eval()
    y_true_all, y_pred_all = [], []
    for b, batch in enumerate(dl):
        if max_batches is not None and b >= max_batches:
            break
        # batch can be (x,) or x
        x_true = batch[0] if isinstance(batch, (list, tuple)) else batch
        x_true = x_true.to(device, non_blocking=True)

        x_masked, mask = make_mask(x_true, mask_ratio=mask_ratio, seed=seed + b)

        pred = model_forward_pred(model, x_masked)  # unified call below

        if mask.sum() > 0:
            y_true_all.append(x_true[mask].detach().cpu().numpy())
            y_pred_all.append(pred[mask].detach().cpu().numpy())

    y_true = np.concatenate(y_true_all)
    y_pred = np.concatenate(y_pred_all)

    mse = mean_squared_error(y_true, y_pred)
    rmse = float(np.sqrt(mse))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred) if np.std(y_true) > 1e-12 else np.nan
    pear = pearsonr(y_true, y_pred)[0]

    print("Evaluation Metrics (on masked positions):")
    print(f"  MSE: {mse:.4f}")
    print(f"  RMSE: {rmse:.4f}")
    print(f"  MAE: {mae:.4f}")
    print(f"  R^2: {r2:.4f}" if not np.isnan(r2) else "  R^2: N/A")
    print(f"  Pearson Correlation: {pear:.4f}")

    return {"mse": mse, "rmse": rmse, "mae": mae, "r2": r2, "pearson": pear}

# --- unify model output: some models return (pred, emb), some just pred
def model_forward_pred(model, x_masked):
    out = model(x_masked)
    if isinstance(out, (list, tuple)):
        return out[0]  # pred
    return out

def train_loop(
    model,
    dl,
    epochs=10,
    mask_ratio=0.15,
    lr=3e-4,
    weight_decay=1e-2,
    accumulation_steps=4,
    grad_clip=1.0,
    use_huber=True,
):
    model.to(device)
    model.train()

    opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay, betas=(0.9, 0.95))

    use_amp = torch.cuda.is_available()
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    huber = nn.SmoothL1Loss(reduction="mean")

    for epoch in range(epochs):
        model.train()
        opt.zero_grad(set_to_none=True)

        running = 0.0
        updates = 0

        for step, batch in enumerate(dl):
            x_true = batch[0] if isinstance(batch, (list, tuple)) else batch
            x_true = x_true.to(device, non_blocking=True)

            x_masked, mask = make_mask(x_true, mask_ratio=mask_ratio)

            with torch.cuda.amp.autocast(enabled=use_amp):
                pred = model_forward_pred(model, x_masked)
                if use_huber:
                    loss = huber(pred[mask], x_true[mask]) if mask.sum() > 0 else huber(pred, x_true)
                else:
                    loss = masked_mse(pred, x_true, mask)
                loss = loss / accumulation_steps

            scaler.scale(loss).backward()

            if (step + 1) % accumulation_steps == 0:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

                scaler.step(opt)
                scaler.update()
                opt.zero_grad(set_to_none=True)

                running += loss.item() * accumulation_steps
                updates += 1

        print(f"Epoch {epoch+1}/{epochs} | Loss: {running/max(1,updates):.4f}")

    return model

#Vanilla Transformer Encoder (baseline)

In [ ]:
class VanillaTransformer(nn.Module):
    def __init__(self, num_genes, d_model=128, nhead=8, num_layers=4, dropout=0.1):
        super().__init__()
        self.num_genes = num_genes
        self.d_model = d_model

        self.gene_emb = nn.Embedding(num_genes, d_model)
        self.val_proj = nn.Linear(1, d_model)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=4*d_model,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.out = nn.Linear(d_model, 1)

    def forward(self, x):  # x: (B,G)
        B, G = x.shape
        gene_ids = torch.arange(G, device=x.device).unsqueeze(0).expand(B, G)
        h = self.val_proj(x.unsqueeze(-1)) + self.gene_emb(gene_ids)  # (B,G,d)
        h = self.encoder(h)                                           # (B,G,d)
        pred = self.out(h).squeeze(-1)                                # (B,G)
        return pred

# ===== train + eval =====
num_genes = X_hvg.shape[1]   # must match your dl tensor dimension
vanilla = VanillaTransformer(num_genes=num_genes, d_model=128, nhead=8, num_layers=4).to(device)

vanilla = train_loop(
    vanilla, dl,
    epochs=30,
    mask_ratio=0.15,
    lr=5e-4,
    weight_decay=1e-2,
    accumulation_steps=4,
    grad_clip=1.0,
    use_huber=True
)

metrics_vanilla = eval_masked_metrics(vanilla, dl, mask_ratio=0.15, seed=42, max_batches=None)


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
/tmp/ipython-input-2522880647.py:90: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
/tmp/ipython-input-2522880647.py:107: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


Epoch 1/30 | Loss: 0.1661
Epoch 2/30 | Loss: 0.1052
Epoch 3/30 | Loss: 0.1018
Epoch 4/30 | Loss: 0.1007


/tmp/ipython-input-2522880647.py:107: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


Epoch 5/30 | Loss: 0.1000
Epoch 6/30 | Loss: 0.0993
Epoch 7/30 | Loss: 0.0990
Epoch 8/30 | Loss: 0.0989
Epoch 9/30 | Loss: 0.0973
Epoch 10/30 | Loss: 0.0960
Epoch 11/30 | Loss: 0.0940
Epoch 12/30 | Loss: 0.0925
Epoch 13/30 | Loss: 0.0915
Epoch 14/30 | Loss: 0.0917
Epoch 15/30 | Loss: 0.0906
Epoch 16/30 | Loss: 0.0900
Epoch 17/30 | Loss: 0.0886
Epoch 18/30 | Loss: 0.0878
Epoch 19/30 | Loss: 0.0871
Epoch 20/30 | Loss: 0.0873
Epoch 21/30 | Loss: 0.0864
Epoch 22/30 | Loss: 0.0851
Epoch 23/30 | Loss: 0.0848
Epoch 24/30 | Loss: 0.0842
Epoch 25/30 | Loss: 0.0830
Epoch 26/30 | Loss: 0.0827
Epoch 27/30 | Loss: 0.0822
Epoch 28/30 | Loss: 0.0823
Epoch 29/30 | Loss: 0.0813
Epoch 30/30 | Loss: 0.0812
Evaluation Metrics (on masked positions):
  MSE: 0.1628
  RMSE: 0.4035
  MAE: 0.3190
  R^2: 0.6874
  Pearson Correlation: 0.8292


#scBERT-style (ранги/дискретизация + Transformer)

In [ ]:
class ScBERTStyle(nn.Module):
    def __init__(self, num_genes, n_bins=32, d_model=128, nhead=8, num_layers=4, dropout=0.1):
        super().__init__()
        self.num_genes = num_genes
        self.n_bins = n_bins
        self.d_model = d_model

        self.gene_emb = nn.Embedding(num_genes, d_model)
        self.bin_emb = nn.Embedding(n_bins + 1, d_model)  # 0..n_bins, include 0 for masked/empty

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=4*d_model,
            dropout=dropout, batch_first=True, activation="gelu", norm_first=True
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.out = nn.Linear(d_model, 1)

    def discretize(self, x):
        # x: (B,G) float, assumes non-negative (log1p)
        # simple per-batch binning by clamping to a max and scaling
        # better: precompute global bin edges on train set
        x_clamped = torch.clamp(x, 0.0, 6.0)  # log1p values usually in this range; adjust if needed
        bins = torch.floor(x_clamped / 6.0 * self.n_bins).long()  # 0..n_bins-1
        bins = torch.clamp(bins, 0, self.n_bins)
        return bins

    def forward(self, x):  # x: (B,G) float masked input
        B, G = x.shape
        gene_ids = torch.arange(G, device=x.device).unsqueeze(0).expand(B, G)
        bins = self.discretize(x)                                  # (B,G) int

        h = self.gene_emb(gene_ids) + self.bin_emb(bins)           # (B,G,d)
        h = self.encoder(h)                                        # (B,G,d)
        pred = self.out(h).squeeze(-1)                             # (B,G) regression
        return pred

# ===== train + eval =====
scbert_style = ScBERTStyle(num_genes=num_genes, n_bins=32, d_model=128, nhead=8, num_layers=4).to(device)

scbert_style = train_loop(
    scbert_style, dl,
    epochs=30,
    mask_ratio=0.15,
    lr=5e-4,
    weight_decay=1e-2,
    accumulation_steps=4,
    grad_clip=1.0,
    use_huber=True
)

metrics_scbert = eval_masked_metrics(scbert_style, dl, mask_ratio=0.15, seed=42, max_batches=None)


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
/tmp/ipython-input-2522880647.py:90: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
/tmp/ipython-input-2522880647.py:107: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


Epoch 1/30 | Loss: 0.1717
Epoch 2/30 | Loss: 0.1040
Epoch 3/30 | Loss: 0.1021
Epoch 4/30 | Loss: 0.1013
Epoch 5/30 | Loss: 0.1005
Epoch 6/30 | Loss: 0.1002
Epoch 7/30 | Loss: 0.0984
Epoch 8/30 | Loss: 0.0959
Epoch 9/30 | Loss: 0.0939
Epoch 10/30 | Loss: 0.0926
Epoch 11/30 | Loss: 0.0908
Epoch 12/30 | Loss: 0.0907
Epoch 13/30 | Loss: 0.0894
Epoch 14/30 | Loss: 0.0890
Epoch 15/30 | Loss: 0.0880
Epoch 16/30 | Loss: 0.0879
Epoch 17/30 | Loss: 0.0870
Epoch 18/30 | Loss: 0.0863
Epoch 19/30 | Loss: 0.0861
Epoch 20/30 | Loss: 0.0853
Epoch 21/30 | Loss: 0.0846
Epoch 22/30 | Loss: 0.0838
Epoch 23/30 | Loss: 0.0834
Epoch 24/30 | Loss: 0.0830
Epoch 25/30 | Loss: 0.0828
Epoch 26/30 | Loss: 0.0820
Epoch 27/30 | Loss: 0.0819
Epoch 28/30 | Loss: 0.0815
Epoch 29/30 | Loss: 0.0816
Epoch 30/30 | Loss: 0.0807
Evaluation Metrics (on masked positions):
  MSE: 0.1634
  RMSE: 0.4042
  MAE: 0.3199
  R^2: 0.6861
  Pearson Correlation: 0.8289


#scGPT-style (gene tokens + value embedding + [CLS] + prediction head)

In [ ]:
class ScGPTStyle(nn.Module):
    def __init__(self, num_genes, d_model=128, nhead=8, num_layers=6, dropout=0.1):
        super().__init__()
        self.num_genes = num_genes
        self.d_model = d_model

        self.gene_emb = nn.Embedding(num_genes, d_model)
        self.val_proj = nn.Linear(1, d_model)
        self.cls = nn.Parameter(torch.zeros(1, 1, d_model))
        self.mask_token = nn.Parameter(torch.zeros(1, 1, d_model))

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=4*d_model,
            dropout=dropout, batch_first=True, activation="gelu", norm_first=True
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.out = nn.Linear(d_model, 1)

        nn.init.normal_(self.cls, std=0.02)
        nn.init.normal_(self.mask_token, std=0.02)

    def forward(self, x):  # x: (B,G)
        B, G = x.shape
        gene_ids = torch.arange(G, device=x.device).unsqueeze(0).expand(B, G)

        val = self.val_proj(x.unsqueeze(-1))              # (B,G,d)
        pos = self.gene_emb(gene_ids)                     # (B,G,d)
        h = val + pos                                     # (B,G,d)

        cls = self.cls.expand(B, 1, self.d_model)
        h = torch.cat([cls, h], dim=1)                    # (B,G+1,d)

        h = self.encoder(h)
        pred = self.out(h[:, 1:, :]).squeeze(-1)          # (B,G)
        cell_emb = h[:, 0, :]
        return pred, cell_emb

# ===== train + eval =====
scgpt_style = ScGPTStyle(num_genes=num_genes, d_model=128, nhead=8, num_layers=6).to(device)

scgpt_style = train_loop(
    scgpt_style, dl,
    epochs=30,
    mask_ratio=0.15,
    lr=5e-4,
    weight_decay=1e-2,
    accumulation_steps=4,
    grad_clip=1.0,
    use_huber=True
)

metrics_scgpt = eval_masked_metrics(scgpt_style, dl, mask_ratio=0.15, seed=42, max_batches=None)


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
/tmp/ipython-input-2522880647.py:90: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
/tmp/ipython-input-2522880647.py:107: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


Epoch 1/30 | Loss: 0.2040
Epoch 2/30 | Loss: 0.1037
Epoch 3/30 | Loss: 0.1016
Epoch 4/30 | Loss: 0.1009
Epoch 5/30 | Loss: 0.0998
Epoch 6/30 | Loss: 0.1000
Epoch 7/30 | Loss: 0.0991
Epoch 8/30 | Loss: 0.0984
Epoch 9/30 | Loss: 0.0965
Epoch 10/30 | Loss: 0.0925
Epoch 11/30 | Loss: 0.0910
Epoch 12/30 | Loss: 0.0882
Epoch 13/30 | Loss: 0.0876
Epoch 14/30 | Loss: 0.0875
Epoch 15/30 | Loss: 0.0862
Epoch 16/30 | Loss: 0.0850
Epoch 17/30 | Loss: 0.0838
Epoch 18/30 | Loss: 0.0835
Epoch 19/30 | Loss: 0.0829
Epoch 20/30 | Loss: 0.0822
Epoch 21/30 | Loss: 0.0813
Epoch 22/30 | Loss: 0.0810
Epoch 23/30 | Loss: 0.0809
Epoch 24/30 | Loss: 0.0803
Epoch 25/30 | Loss: 0.0801
Epoch 26/30 | Loss: 0.0794
Epoch 27/30 | Loss: 0.0791
Epoch 28/30 | Loss: 0.0790
Epoch 29/30 | Loss: 0.0786
Epoch 30/30 | Loss: 0.0782
Evaluation Metrics (on masked positions):
  MSE: 0.1574
  RMSE: 0.3967
  MAE: 0.3192
  R^2: 0.6981
  Pearson Correlation: 0.8358


In [ ]:
import pandas as pd

rows = [
    {"Model":"VanillaTransformer", **metrics_vanilla},
    {"Model":"scBERT-style", **metrics_scbert},
    {"Model":"scGPT-style", **metrics_scgpt},
]
df = pd.DataFrame(rows)[["Model","mse","rmse","mae","r2","pearson"]]
df = df.sort_values("pearson", ascending=False)
df


,Model,mse,rmse,mae,r2,pearson
2,scGPT-style,0.157365,0.396692,0.319247,0.698070,0.835758
0,VanillaTransformer,0.162832,0.403524,0.319043,0.687444,0.829222
1,scBERT-style,0.163387,0.404212,0.319856,0.686124,0.828923


In [ ]:
# Evaluate the first model (Improved GeneTransformer) using the unified function
metrics_improved = eval_masked_metrics(model, dl, mask_ratio=0.15, seed=42, max_batches=None)

# Combine all metrics
rows = [
    {"Model": "Improved GeneTransformer", **metrics_improved},
    {"Model": "VanillaTransformer", **metrics_vanilla},
    {"Model": "scBERT-style", **metrics_scbert},
    {"Model": "scGPT-style", **metrics_scgpt},
]

# Create and display DataFrame
df_all = pd.DataFrame(rows)[["Model", "mse", "rmse", "mae", "r2", "pearson"]]
df_all = df_all.sort_values("pearson", ascending=False)
display(df_all)

Evaluation Metrics (on masked positions):
  MSE: 0.1825
  RMSE: 0.4272
  MAE: 0.3293
  R^2: 0.6494
  Pearson Correlation: 0.8061


,Model,mse,rmse,mae,r2,pearson
3,scGPT-style,0.157365,0.396692,0.319247,0.698070,0.835758
1,VanillaTransformer,0.162832,0.403524,0.319043,0.687444,0.829222
2,scBERT-style,0.163387,0.404212,0.319856,0.686124,0.828923
0,Improved GeneTransformer,0.182465,0.427159,0.329319,0.649399,0.806146
